##### 项目背景与目标
###### 分析淘宝用户行为,洞察漏斗转化与用户价值分层

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / 'date').is_dir() and (p / 'images').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'date'
IMAGES_DIR = PROJECT_ROOT / 'images'

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

数据读取

In [2]:
df = pd.read_csv(DATA_DIR / 'UserBehavior.csv',header = None, names=['user_id','item_id','category_id','behavior_type','timestamp','date'])
print("\n该用户表的数据信息")
print(df.info())
print(df.head(5))


该用户表的数据信息
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048576 entries, 0 to 1048575
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   user_id        1048576 non-null  int64 
 1   item_id        1048576 non-null  int64 
 2   category_id    1048576 non-null  int64 
 3   behavior_type  1048576 non-null  object
 4   timestamp      1048576 non-null  int64 
 5   date           1048576 non-null  object
dtypes: int64(4), object(2)
memory usage: 48.0+ MB
None
   user_id  item_id  category_id behavior_type   timestamp              date
0  1000169  1328010       959452            pv  1505117799   2017/9/11 16:16
1   112815  2489903      1487591            pv  1509965092   2017/11/6 18:44
2   108233  4336503      3531700            pv  1510320815  2017/11/10 21:33
3   114678  3662082      1080785            pv  1510467580  2017/11/12 14:19
4    12775  2897780       886203            pv  1510526244   2017/11/13 6:37

category_id列异常显示为object 应为int64或其余纯数字类型
日期列也应为timestamp类型

##### 数据清洗与预处理
###### 保留列修复,时间转换,去重检查代码,

In [3]:
#处理时间列  将unix时间戳转化为日期
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = pd.to_datetime(df['date'])
df['date'] = df['date'].dt.normalize()
analysis_start = pd.Timestamp('2017-11-25')
analysis_end = pd.Timestamp('2017-12-03')
df = df[df['date'].between(analysis_start, analysis_end)].copy()
print(f"筛选后的数据量: {len(df)}")
print("时间修改完后的数据类型")
print(df.info())
print(df.head(5))

筛选后的数据量: 1048065
时间修改完后的数据类型
<class 'pandas.core.frame.DataFrame'>
Index: 1048065 entries, 494 to 1048558
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   user_id        1048065 non-null  int64         
 1   item_id        1048065 non-null  int64         
 2   category_id    1048065 non-null  int64         
 3   behavior_type  1048065 non-null  object        
 4   timestamp      1048065 non-null  datetime64[ns]
 5   date           1048065 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(3), object(1)
memory usage: 56.0+ MB
None
     user_id  item_id  category_id behavior_type           timestamp  \
494   114726  2014874      1901483            pv 2017-11-24 16:00:00   
495   108891  4295341      3855599            pv 2017-11-24 16:00:01   
496   127538  1849003      2072473            pv 2017-11-24 16:00:01   
497   104088  4539468      3702593            pv 2017-11-24 16:00:02   
498

In [4]:
#处理重复行
duplicated_rows = df[df.duplicated()]
print(duplicated_rows)
#无重复行

Empty DataFrame
Columns: [user_id, item_id, category_id, behavior_type, timestamp, date]
Index: []


In [5]:
# 数据保存
df.to_parquet(DATA_DIR / 'user_behavior_cleaned_dataset.parquet')